Read Data and Handle Missing Values

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import sklearn as skl
import numpy as np
import seaborn as sns
from pathlib import Path

df = pd.read_csv("diabetic_data.csv")
df.head()

In [ ]:
df.drop(columns=['encounter_id', 'patient_nbr','weight','examide', 'citoglipton'], inplace=True)
# Encounter_id and patient number are irrelevant for the output
# weight is over 90% empty
# examide and citoglipton both have no variance whatsoever
df = df.replace('?', np.nan)
df.head()
df.isnull().sum()
# replacing all '?' with nan so that we can count all the missing values
df.dropna(subset=['race', 'diag_1', 'diag_2', 'diag_3'], inplace=True)
# removing the rows with null values
df.isnull().sum()

In [ ]:
# since the rest of the missing columns are relevant to the output we cannot drop them instead filling each with what its missing
df['payer_code'] = df['payer_code'].fillna('not specified')
df['max_glu_serum'] = df['max_glu_serum'].fillna('Not Measured')
df['A1Cresult'] = df['A1Cresult'].fillna('Not Measured')
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
# check for any missing values
df.drop_duplicates(inplace=True)
df.isnull().sum()


Cleaning + Encoding 

In [ ]:
df['medical_specialty'].value_counts()
# display the medical specialties values to pick the most relevant 10

In [ ]:
top_10_specialties = df['medical_specialty'].value_counts().head(10).index
df.loc[~df['medical_specialty'].isin(top_10_specialties), 'medical_specialty'] = 'Other'

In [ ]:
# find a pattern in a medicine column ex.: metformin
df['metformin'].value_counts()

Mapping 

In [ ]:
# identify all the medicine columns
medication_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
    'miglitol', 'troglitazone', 'tolazamide', 'insulin', 
    'glyburide-metformin', 'glipizide-metformin', 
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
    'metformin-pioglitazone'
]
# since all the medication columns have the same 4 measurements we map them
med_mapping = {'No': 0, 'Down': 1, 'Steady': 2, 'Up': 3}
df['metformin'] = df['metformin'].map(med_mapping)

for col in medication_columns:
    df[col] = df[col].map(med_mapping)

In [ ]:
age_mapping = {
    '[0-10)': 0,
    '[10-20)': 1,
    '[20-30)': 2,
    '[30-40)': 3,
    '[40-50)': 4,
    '[50-60)': 5,
    '[60-70)': 6,
    '[70-80)': 7,
    '[80-90)': 8,
    '[90-100)': 9
}

df['age'] = df['age'].map(age_mapping)

In [ ]:
# map these test results into 3 categories
a1c_mapping = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
glu_mapping = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}

df['A1Cresult'] = df['A1Cresult'].map(a1c_mapping)
df['max_glu_serum'] = df['max_glu_serum'].map(glu_mapping)

In [ ]:
# Map 'change' (No change vs. Change)
change_mapping = {'No': 0, 'Ch': 1}
df['change'] = df['change'].map(change_mapping)

# Map 'diabetesMed' (No medication vs. Yes medication)
diab_med_mapping = {'No': 0, 'Yes': 1}
df['diabetesMed'] = df['diabetesMed'].map(diab_med_mapping)

In [ ]:
# After looking up what each diagnosis code means we decided to narrow it down to 9 major diseases
# making it easier to categorize rather than it being continuous values that the model wont understand then later on we encode them
def group_diagnosis(code):
    
    code = str(code).upper()
    
    # V and E codes go into the 'Other' bucket
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    
    # Group the numeric codes by their ICD-9 disease ranges
    try:
        num = float(code)
        if 250 <= num < 251:
            return 'Diabetes'
        elif (390 <= num <= 459) or num == 785:
            return 'Circulatory'
        elif (460 <= num <= 519) or num == 786:
            return 'Respiratory'
        elif (520 <= num <= 579) or num == 787:
            return 'Digestive'
        elif 800 <= num <= 999:
            return 'Injury'
        elif 710 <= num <= 739:
            return 'Musculoskeletal'
        elif (580 <= num <= 629) or num == 788:
            return 'Genitourinary'
        elif 140 <= num <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

# Apply this function to all three diagnosis columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(group_diagnosis)

In [ ]:
#final result mapping the supposed output into 3 classes
multi_class_mapping = {'NO': 0, '>30': 1, '<30': 2}
df['readmitted'] = df['readmitted'].map(multi_class_mapping)

One Hot Encoding

In [ ]:
# final step for categorical data / nominal data to make sense
nominal_columns = ['race', 'gender', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']
df = pd.get_dummies(df, columns=nominal_columns, dtype=int)
df.head()


In [ ]:
#check that everything is numerical + check number of columns
df.info()

In [ ]:
# The 9 categories we created earlier will be mapped into 9 columns instead of 27 in order to reduce the total number of columns
disease_categories = ['Diabetes', 'Circulatory', 'Respiratory', 'Digestive', 
                      'Injury', 'Musculoskeletal', 'Genitourinary', 'Neoplasms', 
                      'Other']

for disease in disease_categories:
    cols_to_check = [f'diag_1_{disease}', f'diag_2_{disease}', f'diag_3_{disease}']
    
    # Valid columns condition to fix an error with pandas libarary
    valid_cols = [col for col in cols_to_check if col in df.columns]
    
    if valid_cols:
        df[f'has_{disease}'] = df[valid_cols].max(axis=1)
        df.drop(columns=valid_cols, inplace=True)

In [ ]:
df.head()